In [1]:
# Cell 1: Imports and project paths
import pandas as pd
import requests
import pycountry
from pathlib import Path

# The notebook lives in notebooks/, so the repo root is one level up.
REPO_ROOT = Path.cwd().parent
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"

# Make sure the raw cache folder exists. The clean CSVs go directly in data/.
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Make DataFrame inspection easier
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print(f"Repo root:  {REPO_ROOT}")
print(f"Data dir:   {DATA_DIR}")
print(f"Raw cache:  {RAW_DIR}")

Repo root:  c:\Users\Vinod\OneDrive\Documents\Masters of Data Science and Innovation (UTS)\36104 Data Visualisation and Narratives\Assignment 3\Bullets-Over-Hugs
Data dir:   c:\Users\Vinod\OneDrive\Documents\Masters of Data Science and Innovation (UTS)\36104 Data Visualisation and Narratives\Assignment 3\Bullets-Over-Hugs\data
Raw cache:  c:\Users\Vinod\OneDrive\Documents\Masters of Data Science and Innovation (UTS)\36104 Data Visualisation and Narratives\Assignment 3\Bullets-Over-Hugs\data\raw


In [2]:
# Cell 2: OWID dataset URLs
# OWID exposes every chart's underlying data as a CSV at the chart URL + ".csv".
# We're pulling three datasets, all based on the UCDP/PRIO conflict data:
#   1. Total deaths per country-year — main table for map circles
#   2. Civilian vs. combatant breakdown — adds detail by victim type
#   3. By conflict type (state-based, non-state, one-sided) — adds detail by conflict type

OWID_SOURCES = {
    "deaths_by_country": "https://ourworldindata.org/grapher/deaths-in-armed-conflicts-by-country.csv",
    "deaths_civ_comb":   "https://ourworldindata.org/grapher/civilian-and-combatant-deaths-in-armed-conflicts-based-on-where-they-occurred.csv",
    "deaths_by_type":    "https://ourworldindata.org/grapher/deaths-in-armed-conflicts-by-type-and-country.csv",
}

OWID_SOURCES

{'deaths_by_country': 'https://ourworldindata.org/grapher/deaths-in-armed-conflicts-by-country.csv',
 'deaths_civ_comb': 'https://ourworldindata.org/grapher/civilian-and-combatant-deaths-in-armed-conflicts-based-on-where-they-occurred.csv',
 'deaths_by_type': 'https://ourworldindata.org/grapher/deaths-in-armed-conflicts-by-type-and-country.csv'}

In [3]:
# Cell 3: Download raw CSVs and cache them locally
def fetch_owid(name: str, url: str) -> pd.DataFrame:
    """Fetch an OWID CSV, cache it to data/raw/, and return as a DataFrame.

    On re-run, uses the cached copy instead of re-downloading.
    """
    cache_path = RAW_DIR / f"{name}.csv"
    if cache_path.exists():
        print(f"  ✓ Using cached {cache_path.name}")
    else:
        print(f"  ↓ Downloading {url}")
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        cache_path.write_bytes(r.content)
        print(f"  ✓ Saved to {cache_path.name}")
    return pd.read_csv(cache_path)

# Fetch all three
raw = {name: fetch_owid(name, url) for name, url in OWID_SOURCES.items()}

# Quick shape check so we know what we're working with
print()
for name, df in raw.items():
    print(f"{name:25s} shape={df.shape}")
    print(f"{'':25s} cols={list(df.columns)}")
    print()

  ↓ Downloading https://ourworldindata.org/grapher/deaths-in-armed-conflicts-by-country.csv
  ✓ Saved to deaths_by_country.csv
  ↓ Downloading https://ourworldindata.org/grapher/civilian-and-combatant-deaths-in-armed-conflicts-based-on-where-they-occurred.csv
  ✓ Saved to deaths_civ_comb.csv
  ↓ Downloading https://ourworldindata.org/grapher/deaths-in-armed-conflicts-by-type-and-country.csv
  ✓ Saved to deaths_by_type.csv

deaths_by_country         shape=(7844, 4)
                          cols=['Entity', 'Code', 'Year', 'Deaths in armed conflicts based on where they occurred']

deaths_civ_comb           shape=(7632, 6)
                          cols=['Entity', 'Code', 'Year', 'Civilian deaths', 'Unclear deaths', 'Combatant deaths']

deaths_by_type            shape=(7632, 7)
                          cols=['Entity', 'Code', 'Year', 'One-sided violence', 'Non-state', 'Intrastate', 'Interstate']



In [4]:
# Cell 4: Quick inspection — what does the data actually look like?

print("=" * 70)
print("deaths_by_country — head and tail")
print("=" * 70)
display(raw["deaths_by_country"].head(3))
display(raw["deaths_by_country"].tail(3))

print("\n" + "=" * 70)
print("Year range across datasets")
print("=" * 70)
for name, df in raw.items():
    print(f"{name:25s} years: {df['Year'].min()} – {df['Year'].max()}")

print("\n" + "=" * 70)
print("Unique 'Entity' values that don't have an ISO Code (likely aggregates)")
print("=" * 70)
no_code = raw["deaths_by_country"]
no_code = no_code[no_code["Code"].isna()]["Entity"].unique()
print(f"{len(no_code)} entities without ISO codes:")
print(sorted(no_code))

print("\n" + "=" * 70)
print("Sample of countries WITH ISO codes (first 10)")
print("=" * 70)
with_code = raw["deaths_by_country"]
with_code = with_code[with_code["Code"].notna()][["Entity", "Code"]].drop_duplicates().sort_values("Entity")
print(with_code.head(10).to_string(index=False))

deaths_by_country — head and tail


,Entity,Code,Year,Deaths in armed conflicts based on where they occurred
0,Abkhazia,OWID_ABK,1989,0
1,Abkhazia,OWID_ABK,1990,0
2,Abkhazia,OWID_ABK,1991,0


,Entity,Code,Year,Deaths in armed conflicts based on where they occurred
7841,Zimbabwe,ZWE,2023,0
7842,Zimbabwe,ZWE,2024,0
7843,Zimbabwe,ZWE,2025,0



Year range across datasets
deaths_by_country         years: 1989 – 2025
deaths_civ_comb           years: 1989 – 2024
deaths_by_type            years: 1989 – 2024

Unique 'Entity' values that don't have an ISO Code (likely aggregates)
5 entities without ISO codes:
['Abyei', 'Americas', 'Asia and Oceania', 'Micronesia', 'Middle East']

Sample of countries WITH ISO codes (first 10)
             Entity     Code
           Abkhazia OWID_ABK
        Afghanistan      AFG
             Africa OWID_AFR
            Albania      ALB
            Algeria      DZA
            Andorra      AND
             Angola      AGO
Antigua and Barbuda      ATG
          Argentina      ARG
            Armenia      ARM


In [5]:
# Cell 5: Investigate edge cases before cleaning

# 1. What does 2024 and 2025 look like in the main dataset? Partial?
print("=" * 70)
print("Row counts per year in deaths_by_country (last 5 years)")
print("=" * 70)
recent = raw["deaths_by_country"].copy()
print(recent.groupby("Year").size().tail(5))

# 2. Which entities have OWID_ prefix codes (i.e. region/special aggregates)?
print("\n" + "=" * 70)
print("Entities with 'OWID_' prefixed codes (these are aggregates, not countries)")
print("=" * 70)
owid_codes = raw["deaths_by_country"][
    raw["deaths_by_country"]["Code"].fillna("").str.startswith("OWID_")
][["Entity", "Code"]].drop_duplicates().sort_values("Entity")
print(owid_codes.to_string(index=False))

# 3. Look at the 5 "no-code" entities in detail
print("\n" + "=" * 70)
print("Rows for the 5 entities without ISO codes (sample)")
print("=" * 70)
no_code_df = raw["deaths_by_country"][raw["deaths_by_country"]["Code"].isna()]
print(no_code_df.groupby("Entity")["Year"].agg(["min", "max", "count"]))

Row counts per year in deaths_by_country (last 5 years)
Year
2021    212
2022    212
2023    212
2024    212
2025    212
dtype: int64

Entities with 'OWID_' prefixed codes (these are aggregates, not countries)
                 Entity     Code
               Abkhazia OWID_ABK
                 Africa OWID_AFR
         Czechoslovakia OWID_CZS
           East Germany OWID_GDR
                 Europe OWID_EUR
                 Kosovo OWID_KOS
          South Ossetia OWID_SOS
           West Germany OWID_GFR
                  World OWID_WRL
Yemen People's Republic OWID_YPR
             Yugoslavia OWID_YGS

Rows for the 5 entities without ISO codes (sample)
                   min   max  count
Entity                             
Abyei             1989  2025     37
Americas          1989  2025     37
Asia and Oceania  1989  2025     37
Micronesia        1989  2025     37
Middle East       1989  2025     37


In [6]:
# Cell 6: Standardise each raw DataFrame
# - Rename columns to snake_case
# - Filter to country-level rows only (drop region aggregates)
# - Filter to 2015–2024 (our scope)
# - Keep Kosovo (OWID_KOS) as an exception since it's a recognised country

YEAR_MIN = 2015
YEAR_MAX = 2024

def is_country_code(code: str) -> bool:
    """Return True if the code represents an actual country.
    
    Real ISO-3 codes are 3 uppercase letters. OWID-prefixed codes are
    aggregates or special entities — we drop those, EXCEPT Kosovo,
    which is a recognised country that simply lacks an ISO assignment.
    """
    if pd.isna(code):
        return False
    if code == "OWID_KOS":  # Kosovo — keep
        return True
    return len(code) == 3 and code.isupper() and code.isalpha()

def standardise(df: pd.DataFrame, value_columns: dict) -> pd.DataFrame:
    """Apply common cleaning steps to an OWID DataFrame.
    
    value_columns: {original_name: clean_snake_case_name}
    """
    df = df.copy()
    
    # Rename Entity/Code/Year + value columns
    rename_map = {"Entity": "country", "Code": "iso3", "Year": "year"}
    rename_map.update(value_columns)
    df = df.rename(columns=rename_map)
    
    # Filter to actual countries
    df = df[df["iso3"].apply(is_country_code)].copy()
    
    # Normalise Kosovo's code to KOS (drop the OWID_ prefix for cleaner output)
    df.loc[df["iso3"] == "OWID_KOS", "iso3"] = "KOS"
    
    # Filter to 2015–2024
    df = df[(df["year"] >= YEAR_MIN) & (df["year"] <= YEAR_MAX)].copy()
    
    # Reset index for cleanliness
    df = df.reset_index(drop=True)
    return df

# Quick test on one dataset
test = standardise(
    raw["deaths_by_country"],
    {"Deaths in armed conflicts based on where they occurred": "total_deaths"},
)
print(f"Shape after cleaning: {test.shape}")
print(f"Year range: {test['year'].min()} – {test['year'].max()}")
print(f"Unique countries: {test['iso3'].nunique()}")
print(f"\nFirst 5 rows:")
display(test.head())
print(f"\nKosovo check (should appear with KOS code):")
display(test[test["iso3"] == "KOS"].head(3))

Shape after cleaning: (1970, 4)
Year range: 2015 – 2024
Unique countries: 197

First 5 rows:


,country,iso3,year,total_deaths
0,Afghanistan,AFG,2015,18016
1,Afghanistan,AFG,2016,18991
2,Afghanistan,AFG,2017,19766
3,Afghanistan,AFG,2018,26849
4,Afghanistan,AFG,2019,30434



Kosovo check (should appear with KOS code):


,country,iso3,year,total_deaths
910,Kosovo,KOS,2015,0
911,Kosovo,KOS,2016,0
912,Kosovo,KOS,2017,0


In [7]:
# Cell 7: Apply standardisation to all three datasets

clean_total = standardise(
    raw["deaths_by_country"],
    {"Deaths in armed conflicts based on where they occurred": "total_deaths"},
)

clean_civ_comb = standardise(
    raw["deaths_civ_comb"],
    {
        "Civilian deaths": "civilian_deaths",
        "Unclear deaths": "unclear_deaths",
        "Combatant deaths": "combatant_deaths",
    },
)

clean_by_type = standardise(
    raw["deaths_by_type"],
    {
        "One-sided violence": "one_sided_violence",
        "Non-state": "non_state",
        "Intrastate": "intrastate",
        "Interstate": "interstate",
    },
)

# Quick check — all should have similar shapes
for name, df in [("total", clean_total), ("civ_comb", clean_civ_comb), ("by_type", clean_by_type)]:
    print(f"{name:10s} shape={df.shape}  countries={df['iso3'].nunique()}  years={df['year'].min()}–{df['year'].max()}")

total      shape=(1970, 4)  countries=197  years=2015–2024
civ_comb   shape=(1970, 6)  countries=197  years=2015–2024
by_type    shape=(1970, 7)  countries=197  years=2015–2024


In [8]:
# Cell 8: Merge all three cleaned datasets into one master table.
# A 1:1 merge on (iso3, year) — country names also match but iso3 is the
# canonical join key. We sort columns to put identifiers first, then totals,
# then breakdowns.

JOIN_KEYS = ["iso3", "year"]

# Start from the total dataset; merge in the breakdowns
master = clean_total.merge(
    clean_civ_comb.drop(columns=["country"]),  # drop duplicate country col
    on=JOIN_KEYS,
    how="left",
    validate="one_to_one",
)
master = master.merge(
    clean_by_type.drop(columns=["country"]),
    on=JOIN_KEYS,
    how="left",
    validate="one_to_one",
)

# Reorder columns logically
master = master[[
    "country", "iso3", "year",
    "total_deaths",
    "civilian_deaths", "combatant_deaths", "unclear_deaths",
    "one_sided_violence", "non_state", "intrastate", "interstate",
]]

print(f"Master shape: {master.shape}")
print(f"Expected:     (1970, 11)")
print(f"\nNulls per column (should all be 0):")
print(master.isna().sum())
print(f"\nFirst 5 rows of a country with conflict (Afghanistan):")
display(master[master["iso3"] == "AFG"].head())

Master shape: (1970, 11)
Expected:     (1970, 11)

Nulls per column (should all be 0):
country               0
iso3                  0
year                  0
total_deaths          0
civilian_deaths       0
combatant_deaths      0
unclear_deaths        0
one_sided_violence    0
non_state             0
intrastate            0
interstate            0
dtype: int64

First 5 rows of a country with conflict (Afghanistan):


,country,iso3,year,total_deaths,civilian_deaths,combatant_deaths,unclear_deaths,one_sided_violence,non_state,intrastate,interstate
0,Afghanistan,AFG,2015,18016,899,16136,981,174,532,17310,0
1,Afghanistan,AFG,2016,18991,1067,17471,453,341,665,17985,0
2,Afghanistan,AFG,2017,19766,1052,18318,396,335,412,19019,0
3,Afghanistan,AFG,2018,26849,1839,24155,855,542,631,25676,0
4,Afghanistan,AFG,2019,30434,1756,28385,293,379,113,29942,0


In [9]:
# Cell 9: Sanity check — do the breakdowns reconcile with totals?
# Both breakdowns (civ/comb/unclear AND state-based/non-state/etc.) should
# each sum to the total. If not, OWID may be reporting them with different
# methodologies and we need to flag this for the dashboard team.

check = master.copy()
check["sum_victim_type"] = check[["civilian_deaths", "combatant_deaths", "unclear_deaths"]].sum(axis=1)
check["sum_conflict_type"] = check[["one_sided_violence", "non_state", "intrastate", "interstate"]].sum(axis=1)

# Look at countries with deaths > 0 to see if reconciliation holds
nonzero = check[check["total_deaths"] > 0].copy()
nonzero["diff_victim"]   = nonzero["sum_victim_type"]   - nonzero["total_deaths"]
nonzero["diff_conflict"] = nonzero["sum_conflict_type"] - nonzero["total_deaths"]

print(f"Rows with total_deaths > 0: {len(nonzero)}")
print(f"\nVictim-type breakdown vs total:")
print(f"  Mean diff:   {nonzero['diff_victim'].mean():>10.2f}")
print(f"  Max diff:    {nonzero['diff_victim'].abs().max():>10.0f}")
print(f"  Rows matching exactly: {(nonzero['diff_victim'] == 0).sum()} / {len(nonzero)}")

print(f"\nConflict-type breakdown vs total:")
print(f"  Mean diff:   {nonzero['diff_conflict'].mean():>10.2f}")
print(f"  Max diff:    {nonzero['diff_conflict'].abs().max():>10.0f}")
print(f"  Rows matching exactly: {(nonzero['diff_conflict'] == 0).sum()} / {len(nonzero)}")

# Show worst mismatches if any
worst = nonzero.reindex(nonzero["diff_victim"].abs().sort_values(ascending=False).index)
print("\nTop 5 largest victim-type discrepancies:")
display(worst[["country", "year", "total_deaths", "sum_victim_type", "diff_victim"]].head())

Rows with total_deaths > 0: 486

Victim-type breakdown vs total:
  Mean diff:         0.00
  Max diff:             0
  Rows matching exactly: 486 / 486

Conflict-type breakdown vs total:
  Mean diff:         0.00
  Max diff:             0
  Rows matching exactly: 486 / 486

Top 5 largest victim-type discrepancies:


,country,year,total_deaths,sum_victim_type,diff_victim
0,Afghanistan,2015,18016,18016,0
1,Afghanistan,2016,18991,18991,0
2,Afghanistan,2017,19766,19766,0
3,Afghanistan,2018,26849,26849,0
4,Afghanistan,2019,30434,30434,0


In [10]:
# Cell 10: Add country centroids for mapping
# Plotly scatter_geo can map by ISO-3 alone, but explicit lat/long gives the
# dashboard team flexibility (pydeck, custom basemaps, country-pair lines etc.).
# We use the public 'country-coord' dataset from the world-bank style centroid file.

CENTROID_URL = (
    "https://raw.githubusercontent.com/google/dspl/master/samples/"
    "google/canonical/countries.csv"
)

centroid_cache = RAW_DIR / "country_centroids.csv"
if not centroid_cache.exists():
    print(f"↓ Downloading {CENTROID_URL}")
    r = requests.get(CENTROID_URL, timeout=30)
    r.raise_for_status()
    centroid_cache.write_bytes(r.content)
    print(f"✓ Saved to {centroid_cache.name}")
else:
    print(f"✓ Using cached {centroid_cache.name}")

centroids = pd.read_csv(centroid_cache)
print(f"Shape: {centroids.shape}")
print(f"Columns: {list(centroids.columns)}")
display(centroids.head())

↓ Downloading https://raw.githubusercontent.com/google/dspl/master/samples/google/canonical/countries.csv
✓ Saved to country_centroids.csv
Shape: (245, 4)
Columns: ['country', 'latitude', 'longitude', 'name']


,country,latitude,longitude,name
0,AD,42.546245,1.601554,Andorra
1,AE,23.424076,53.847818,United Arab Emirates
2,AF,33.939110,67.709953,Afghanistan
3,AG,17.060816,-61.796428,Antigua and Barbuda
4,AI,18.220554,-63.068615,Anguilla


In [11]:
# Cell 11: Convert centroid ISO-2 codes to ISO-3, then join to master.

# 1. Add iso3 to the centroids DataFrame using pycountry
def iso2_to_iso3(iso2: str) -> str | None:
    try:
        return pycountry.countries.get(alpha_2=iso2).alpha_3
    except (AttributeError, LookupError):
        return None

centroids = centroids.rename(columns={"country": "iso2", "name": "country_centroid"})
centroids["iso3"] = centroids["iso2"].apply(iso2_to_iso3)

# 2. Manual entries for non-standard codes we need
# Kosovo declared independence in 2008. ISO hasn't assigned it a code, but
# OWID uses OWID_KOS (which we cleaned to "KOS"). Coordinates: ~42.6°N, 20.9°E.
manual_centroids = pd.DataFrame([
    {"iso2": None, "iso3": "KOS", "latitude": 42.6026, "longitude": 20.9030, "country_centroid": "Kosovo"},
])
centroids = pd.concat([centroids, manual_centroids], ignore_index=True)

# 3. How many of our 197 countries have a matching centroid?
master_iso3 = set(master["iso3"].unique())
centroid_iso3 = set(centroids["iso3"].dropna().unique())
missing = master_iso3 - centroid_iso3
print(f"Master countries: {len(master_iso3)}")
print(f"Centroid countries: {len(centroid_iso3)}")
print(f"Missing centroids for: {len(missing)} countries")
if missing:
    print(f"  → {sorted(missing)}")
    # Show what these are in our master
    print("\nNames of missing countries:")
    print(master[master["iso3"].isin(missing)][["country", "iso3"]].drop_duplicates().to_string(index=False))

Master countries: 197
Centroid countries: 242
Missing centroids for: 2 countries
  → ['NAM', 'SSD']

Names of missing countries:
    country iso3
    Namibia  NAM
South Sudan  SSD


In [12]:
# Diagnostic: was Namibia silently turned into NaN?
print("Centroid rows where iso2 is null:")
display(centroids[centroids["iso2"].isna()])

Centroid rows where iso2 is null:


,iso2,latitude,longitude,country_centroid,iso3
156,NaN,-22.95764,18.49041,Namibia,NaN
245,None,42.60260,20.90300,Kosovo,KOS


In [13]:
# Cell 12: Patch missing centroids
# - Namibia: ISO-2 "NA" was likely auto-converted to NaN by pandas read_csv
# - South Sudan: gained independence in 2011, missing from older centroid datasets

extra_centroids = pd.DataFrame([
    {"iso2": "NA",  "iso3": "NAM", "latitude": -22.9576, "longitude": 18.4904, "country_centroid": "Namibia"},
    {"iso2": "SS",  "iso3": "SSD", "latitude":   6.8770, "longitude": 31.3070, "country_centroid": "South Sudan"},
])
centroids = pd.concat([centroids, extra_centroids], ignore_index=True)

# Re-check coverage
centroid_iso3 = set(centroids["iso3"].dropna().unique())
missing = set(master["iso3"].unique()) - centroid_iso3
print(f"Missing centroids: {len(missing)}")
assert not missing, f"Still missing: {missing}"
print("✓ All 197 countries have centroids.")

Missing centroids: 0
✓ All 197 countries have centroids.


In [14]:
# Cell 13: Join centroids onto master.
# Use a left join — every master row should find a match (we just verified).

centroid_lookup = centroids[["iso3", "latitude", "longitude"]].drop_duplicates(subset="iso3")

master = master.merge(centroid_lookup, on="iso3", how="left", validate="many_to_one")

# Final integrity check
print(f"Master shape: {master.shape}  (expected (1970, 13))")
print(f"\nNulls in lat/long (should be 0):")
print(master[["latitude", "longitude"]].isna().sum())
print(f"\nFinal columns: {list(master.columns)}")
display(master.head(3))

Master shape: (1970, 13)  (expected (1970, 13))

Nulls in lat/long (should be 0):
latitude     0
longitude    0
dtype: int64

Final columns: ['country', 'iso3', 'year', 'total_deaths', 'civilian_deaths', 'combatant_deaths', 'unclear_deaths', 'one_sided_violence', 'non_state', 'intrastate', 'interstate', 'latitude', 'longitude']


,country,iso3,year,total_deaths,civilian_deaths,combatant_deaths,unclear_deaths,one_sided_violence,non_state,intrastate,interstate,latitude,longitude
0,Afghanistan,AFG,2015,18016,899,16136,981,174,532,17310,0,33.93911,67.709953
1,Afghanistan,AFG,2016,18991,1067,17471,453,341,665,17985,0,33.93911,67.709953
2,Afghanistan,AFG,2017,19766,1052,18318,396,335,412,19019,0,33.93911,67.709953


In [15]:
# Cell 14: Table 1 — country-year main table
# This is the primary table for the dashboard's bubble map + year slider.
# One row per country-year, with totals and lat/long.

table1_country_year = master[[
    "country", "iso3", "year",
    "latitude", "longitude",
    "total_deaths",
    "civilian_deaths", "combatant_deaths", "unclear_deaths",
]].copy()

# Sort for predictability — alphabetical by country, then chronological
table1_country_year = table1_country_year.sort_values(["country", "year"]).reset_index(drop=True)

print(f"Shape: {table1_country_year.shape}")
print(f"\nDtypes:")
print(table1_country_year.dtypes)
print(f"\nSample (Yemen — high-conflict country):")
display(table1_country_year[table1_country_year["iso3"] == "YEM"])

Shape: (1970, 9)

Dtypes:
country                 str
iso3                    str
year                  int64
latitude            float64
longitude           float64
total_deaths          int64
civilian_deaths       int64
combatant_deaths      int64
unclear_deaths        int64
dtype: object

Sample (Yemen — high-conflict country):


,country,iso3,year,latitude,longitude,total_deaths,civilian_deaths,combatant_deaths,unclear_deaths
1940,Yemen,YEM,2015,15.552727,48.516388,8102,2282,4081,1739
1941,Yemen,YEM,2016,15.552727,48.516388,4021,854,2513,654
1942,Yemen,YEM,2017,15.552727,48.516388,3597,836,2371,390
1943,Yemen,YEM,2018,15.552727,48.516388,4982,726,2969,1287
1944,Yemen,YEM,2019,15.552727,48.516388,2925,657,1863,405
1945,Yemen,YEM,2020,15.552727,48.516388,6733,693,5070,970
1946,Yemen,YEM,2021,15.552727,48.516388,23441,707,22045,689
1947,Yemen,YEM,2022,15.552727,48.516388,3176,767,2340,69
1948,Yemen,YEM,2023,15.552727,48.516388,902,288,600,14
1949,Yemen,YEM,2024,15.552727,48.516388,980,151,643,186


In [16]:
# Cell 15: Table 2 — conflict-type breakdown in long format
# Long format makes Plotly stacked bars / treemaps trivial.
# One row per country-year-conflict_type.

# Melt the four conflict-type columns into long format
type_cols = ["one_sided_violence", "non_state", "intrastate", "interstate"]
table2_by_type = master.melt(
    id_vars=["country", "iso3", "year", "latitude", "longitude"],
    value_vars=type_cols,
    var_name="conflict_type",
    value_name="deaths",
)

# Make conflict_type values display-friendly
type_labels = {
    "one_sided_violence": "One-sided violence",
    "non_state":          "Non-state",
    "intrastate":         "Intrastate",
    "interstate":         "Interstate",
}
table2_by_type["conflict_type_label"] = table2_by_type["conflict_type"].map(type_labels)

# Sort: country, year, then conflict type in a sensible logical order
type_order = ["interstate", "intrastate", "non_state", "one_sided_violence"]
table2_by_type["conflict_type"] = pd.Categorical(
    table2_by_type["conflict_type"], categories=type_order, ordered=True
)
table2_by_type = table2_by_type.sort_values(
    ["country", "year", "conflict_type"]
).reset_index(drop=True)

# Convert back to plain string after sorting (cleaner CSV output)
table2_by_type["conflict_type"] = table2_by_type["conflict_type"].astype(str)

print(f"Shape: {table2_by_type.shape}  (expected: 1970 × 4 = {1970*4})")
print(f"\nConflict types:")
print(table2_by_type["conflict_type"].value_counts())
print(f"\nSample — Ukraine 2022 (year of full-scale invasion):")
display(table2_by_type[(table2_by_type["iso3"] == "UKR") & (table2_by_type["year"] == 2022)])

Shape: (7880, 8)  (expected: 1970 × 4 = 7880)

Conflict types:
conflict_type
interstate            1970
intrastate            1970
non_state             1970
one_sided_violence    1970
Name: count, dtype: int64

Sample — Ukraine 2022 (year of full-scale invasion):


,country,iso3,year,latitude,longitude,conflict_type,deaths,conflict_type_label
7388,Ukraine,UKR,2022,48.379433,31.16558,interstate,91435,Interstate
7389,Ukraine,UKR,2022,48.379433,31.16558,intrastate,43,Intrastate
7390,Ukraine,UKR,2022,48.379433,31.16558,non_state,0,Non-state
7391,Ukraine,UKR,2022,48.379433,31.16558,one_sided_violence,1130,One-sided violence


In [18]:
# Cell 16: Table 3 — country summary (one row per country, 2015–2024 totals)

table3_summary = (
    master.groupby(["country", "iso3", "latitude", "longitude"], as_index=False)
    .agg(
        total_deaths_2015_2024     = ("total_deaths",       "sum"),
        civilian_deaths_2015_2024  = ("civilian_deaths",    "sum"),
        combatant_deaths_2015_2024 = ("combatant_deaths",   "sum"),
        peak_year_deaths           = ("total_deaths",       "max"),
        years_with_conflict        = ("total_deaths",       lambda s: int((s > 0).sum())),
    )
)

# Add: which year was the peak?
peak_year = (
    master.sort_values("total_deaths", ascending=False)
    .drop_duplicates("iso3")
    [["iso3", "year"]]
    .rename(columns={"year": "peak_year"})
)
table3_summary = table3_summary.merge(peak_year, on="iso3", how="left")

# Add: civilian share of deaths (handy for the dashboard's "human cost" framing).
# Use numpy.nan (not pd.NA) for the divide-by-zero guard — pd.NA is object-dtype
# and breaks .round(); np.nan stays in float64 cleanly.
import numpy as np
denom = table3_summary["total_deaths_2015_2024"].replace(0, np.nan)
table3_summary["civilian_share_pct"] = (
    (table3_summary["civilian_deaths_2015_2024"] / denom * 100).round(1)
)

# Sort by total deaths descending — top conflicts surface first
table3_summary = table3_summary.sort_values(
    "total_deaths_2015_2024", ascending=False
).reset_index(drop=True)

# Reorder for readability
table3_summary = table3_summary[[
    "country", "iso3", "latitude", "longitude",
    "total_deaths_2015_2024",
    "civilian_deaths_2015_2024", "combatant_deaths_2015_2024",
    "civilian_share_pct",
    "peak_year", "peak_year_deaths",
    "years_with_conflict",
]]

print(f"Shape: {table3_summary.shape}  (expected (197, 11))")
print(f"\nTop 15 countries by total conflict deaths, 2015–2024:")
display(table3_summary.head(15))

Shape: (197, 11)  (expected (197, 11))

Top 15 countries by total conflict deaths, 2015–2024:


,country,iso3,latitude,longitude,total_deaths_2015_2024,civilian_deaths_2015_2024,combatant_deaths_2015_2024,civilian_share_pct,peak_year,peak_year_deaths,years_with_conflict
0,Ethiopia,ETH,9.145000,40.489673,327642,14267,303486,4.4,2022,164991,10
1,Ukraine,UKR,48.379433,31.165580,240109,24787,197221,10.3,2022,92608,10
2,Syria,SYR,34.802075,38.996815,199674,54689,134143,27.4,2015,59544,10
3,Afghanistan,AFG,33.939110,67.709953,173550,11031,158977,6.4,2021,36375,10
4,Mexico,MEX,23.634501,-102.552784,102687,685,1024,0.7,2021,15020,10
5,Yemen,YEM,15.552727,48.516388,58859,7961,44495,13.5,2021,23441,10
6,Palestine,PSE,31.952162,35.233154,47963,27729,956,57.8,2023,26129,6
7,Iraq,IRQ,33.223191,43.679291,41027,12731,26385,31.0,2016,12038,10
8,Nigeria,NGA,9.081999,8.675277,37622,8489,23887,22.6,2015,9093,10
9,Democratic Republic of Congo,COD,-4.038333,21.758664,34988,19892,9978,56.9,2017,5822,10


In [19]:
# Cell 17: Write all three tables to data/

OUTPUT_PATHS = {
    "country_year": DATA_DIR / "conflict_deaths_country_year.csv",
    "by_type":      DATA_DIR / "conflict_deaths_by_type.csv",
    "summary":      DATA_DIR / "conflict_deaths_summary.csv",
}

table1_country_year.to_csv(OUTPUT_PATHS["country_year"], index=False)
table2_by_type.to_csv(OUTPUT_PATHS["by_type"], index=False)
table3_summary.to_csv(OUTPUT_PATHS["summary"], index=False)

# Verify by reading them back
print("=" * 60)
print("Files written:")
print("=" * 60)
for name, path in OUTPUT_PATHS.items():
    size_kb = path.stat().st_size / 1024
    df_check = pd.read_csv(path)
    print(f"  {path.name:42s}  {df_check.shape[0]:>5} rows  {size_kb:>6.1f} KB")

Files written:
  conflict_deaths_country_year.csv             1970 rows    93.8 KB
  conflict_deaths_by_type.csv                  7880 rows   516.2 KB
  conflict_deaths_summary.csv                   197 rows    10.8 KB


In [20]:
# Cell 18: Generate a data dictionary for the dashboard team
# This documents column meanings, methodology caveats, and known data quirks
# so the team building the visualisations doesn't misinterpret the numbers.

DATA_DICT = """# Conflict Casualties Dataset — Data Dictionary

**Topic 1: Modern Conflicts and Casualties (2015–2024)**

## Source
All data is from [Our World in Data](https://ourworldindata.org/war-and-peace), based on the [UCDP/PRIO Armed Conflict Dataset](https://ucdp.uu.se/) — the standard academic source for armed-conflict statistics.

## Files

### `conflict_deaths_country_year.csv` — main map/timeline table
One row per country-year. Use this for the world bubble map with a year slider.

| Column | Type | Description |
|---|---|---|
| `country` | str | Country name |
| `iso3` | str | ISO 3166-1 alpha-3 code (Kosovo uses `KOS` — non-standard but unambiguous) |
| `year` | int | Year (2015–2024) |
| `latitude` | float | Country centroid latitude (for map plotting) |
| `longitude` | float | Country centroid longitude |
| `total_deaths` | int | Total deaths in armed conflicts that year |
| `civilian_deaths` | int | Subset: deaths classified as civilian |
| `combatant_deaths` | int | Subset: deaths classified as combatant |
| `unclear_deaths` | int | Subset: deaths whose victim type couldn't be classified |

`civilian + combatant + unclear = total_deaths` exactly.

### `conflict_deaths_by_type.csv` — conflict-type breakdown (long format)
One row per country-year-conflict_type. Use this for stacked bars / treemaps showing what *kind* of conflict drives deaths in each country. Long format means it works directly with Plotly's `color="conflict_type"`.

| Column | Type | Description |
|---|---|---|
| `country` | str | Country name |
| `iso3` | str | ISO-3 code |
| `year` | int | Year |
| `latitude` | float | Country centroid latitude |
| `longitude` | float | Country centroid longitude |
| `conflict_type` | str | Machine-friendly: `interstate`, `intrastate`, `non_state`, `one_sided_violence` |
| `deaths` | int | Deaths attributed to this conflict type that year |
| `conflict_type_label` | str | Display-friendly label (e.g. "One-sided violence") |

The four conflict types sum exactly to `total_deaths` in the country-year table.

**Conflict type definitions:**
- **Interstate**: state vs. state (e.g. Russia–Ukraine 2022)
- **Intrastate**: state vs. non-state group within its borders (civil wars, insurgencies)
- **Non-state**: between organised armed groups, no state party (e.g. cartel wars, militia clashes)
- **One-sided violence**: armed group attacking civilians (e.g. ethnic massacres, terrorism)

### `conflict_deaths_summary.csv` — country totals (one row per country)
Use this for top-N rankings, summary cards, and tooltips.

| Column | Type | Description |
|---|---|---|
| `country` | str | Country name |
| `iso3` | str | ISO-3 code |
| `latitude` | float | Country centroid latitude |
| `longitude` | float | Country centroid longitude |
| `total_deaths_2015_2024` | int | Sum of all conflict deaths over the decade |
| `civilian_deaths_2015_2024` | int | Sum of civilian deaths over the decade |
| `combatant_deaths_2015_2024` | int | Sum of combatant deaths over the decade |
| `civilian_share_pct` | float | Civilian deaths as % of total (NaN for countries with zero deaths) |
| `peak_year` | int | The year with the most deaths |
| `peak_year_deaths` | int | Deaths in that peak year |
| `years_with_conflict` | int | Out of 10 years, how many had any deaths |

## ⚠️ Methodology caveats (important for narrative integrity)

1. **UCDP only counts directly battle-related deaths.** Excludes indirect deaths from famine, disease, displacement, etc. So Yemen's UCDP total (~58k) is much lower than civil-society estimates of 150k+ that include indirect causes. The dashboard should note this if displaying Yemen.

2. **Mexico and Brazil**: most deaths are classified as **unclear** rather than civilian or combatant. Don't use civilian-share metrics for these countries without a caveat — the low civilian share (Mexico: 0.7%) is a data artefact, not a finding.

3. **Cumulative civilian share for high-target conflicts**: Palestine (57.8%), Myanmar (58.3%), DRC (56.9%), Syria (27.4%). These are conflicts where civilian targeting is/was a defining feature.

4. **Years with zero deaths** are still included as rows. Most countries are peaceful in most years — that's the point. The data is rectangular: 197 countries × 10 years = 1,970 rows in the country-year table.

5. **Kosovo** uses `KOS` instead of an official ISO-3 code (none has been assigned by ISO). Make sure any country-name lookups in the dashboard handle this.

## Re-running the data preparation

The notebook `notebooks/01_topic1_conflict_casualties.ipynb` produces these files. To re-run:

```bash
pip install requests pycountry  # in addition to the project's requirements.txt
```

Then open the notebook and run all cells. The OWID downloads are cached in `data/raw/` (gitignored) so re-runs are fast.
"""

dict_path = DATA_DIR / "data_dictionary.md"
dict_path.write_text(DATA_DICT, encoding="utf-8")
print(f"✓ Wrote data dictionary to {dict_path.relative_to(REPO_ROOT)}")
print(f"  Size: {dict_path.stat().st_size / 1024:.1f} KB")

✓ Wrote data dictionary to data\data_dictionary.md
  Size: 4.8 KB
